In [ ]:
# 1. Imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

# Display / formatting
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:0.6f}")

DATA_PATH = Path("../data/processed/mru_usd_ohlc_clean.csv")

In [ ]:
# 2. Load cleaned data
print("Loading cleaned MRU/USD data from:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing file: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# Basic quick preview
display(df.head(5))
display(df.tail(5))
print("\nColumn dtypes:")
print(df.dtypes)

In [ ]:
# 3. Basic overview: counts, date range, integrity checks
n_rows = len(df)
n_unique_dates = df["date"].nunique()
date_min = df["date"].min()
date_max = df["date"].max()

print(f"Total rows: {n_rows}")
print(f"Distinct dates: {n_unique_dates}")
print(f"Date range: {date_min.date()} → {date_max.date()}")
print(f"Date sorted ascending? {df['date'].is_monotonic_increasing}")

# Duplicate dates and missing OHLC counts
dup_count = (df["date"].duplicated()).sum()
print(f"Duplicate-date rows: {int(dup_count)}")

ohlc_cols = ["open", "high", "low", "close"]
print("\nMissing values per OHLC column:")
print(df[ohlc_cols].isna().sum())

print("\nOHLC descriptive statistics:")
display(df[ohlc_cols].describe().T)

In [ ]:
# 4. Weekday distribution
df["weekday"] = df["date"].dt.weekday  # 0=Mon, 6=Sun
weekday_counts = df["weekday"].value_counts().sort_index()

weekday_map = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
weekday_summary = pd.DataFrame({
    "weekday": weekday_counts.index,
    "weekday_name": [weekday_map[w] for w in weekday_counts.index],
    "count": weekday_counts.values
})

display(weekday_summary)
has_weekend = bool(weekday_counts.get(5, 0) + weekday_counts.get(6, 0) > 0)
print(f"Weekend rows present (Sat/Sun)? {has_weekend}")

In [ ]:
# 5. Price-level summary and period breakdown
print("Close price summary:")
display(df["close"].describe())

# Rough period definitions (adjust dates to your data if needed)
price_periods = [
    ("2000-01-01", "2009-12-31", "2000–2009"),
    ("2010-01-01", "2014-12-31", "2010–2014"),
    ("2015-01-01", "2019-12-31", "2015–2019"),
    ("2020-01-01", date_max.strftime("%Y-%m-%d"), "2020–end"),
]

rows = []
for start, end, label in price_periods:
    mask = (df["date"] >= pd.to_datetime(start)) & (df["date"] <= pd.to_datetime(end))
    sub = df.loc[mask, "close"]
    if sub.empty:
        continue
    rows.append({
        "period": label,
        "start": start,
        "end": end,
        "count": int(len(sub)),
        "min_close": float(sub.min()),
        "max_close": float(sub.max()),
        "mean_close": float(sub.mean()),
    })

if rows:
    price_periods_df = pd.DataFrame(rows)
    display(price_periods_df)

# Plot close price (interactive)
fig_close = px.line(df, x="date", y="close", title="MRU/USD Close Price (cleaned)")
fig_close.show()

In [ ]:
# 6. Compute returns
print("Computing log and simple returns")

zero_close_count = int((df["close"] == 0).sum())
print(f"Rows with close == 0: {zero_close_count}")

df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["simple_return"] = df["close"].pct_change()

ret = df["log_return"].dropna()
print("\nLog return summary:")
display(ret.describe())

quantiles = ret.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print("\nSelected log-return quantiles:")
display(quantiles)

# Plot log returns
fig_ret = px.line(df, x="date", y="log_return", title="Daily Log Returns")
fig_ret.show()

In [ ]:
# 7. Returns by year and era
yearly_stats = (
    df.dropna(subset=["log_return"])
      .groupby(df["date"].dt.year)["log_return"]
      .agg(["count", "mean", "std", "min", "max"])
      .reset_index()
      .rename(columns={"date": "year"})
)

display(yearly_stats.head(20))

# assign simple eras for structural overview
def assign_era(y):
    if y <= 2004:
        return "2000–2004"
    if y <= 2009:
        return "2005–2009"
    if y <= 2014:
        return "2010–2014"
    if y <= 2019:
        return "2015–2019"
    return "2020–"

df["era"] = df["date"].dt.year.apply(assign_era)

era_stats = (
    df.dropna(subset=["log_return"])
      .groupby("era")["log_return"]
      .agg(["count", "mean", "std", "min", "max"])
      .reset_index()
)
display(era_stats)

fig_era_vol = px.bar(era_stats, x="era", y="std", title="Std(log return) by era")
fig_era_vol.show()

In [ ]:
# 8. Rolling volatility
window_short = 30
window_long = 90

df["vol_30"] = df["log_return"].rolling(window_short).std()
df["vol_90"] = df["log_return"].rolling(window_long).std()

print("30-day vol (non-NaN) stats:")
display(df["vol_30"].dropna().describe())

print("90-day vol (non-NaN) stats:")
display(df["vol_90"].dropna().describe())

# Save rolling vol (derived)
derived_dir = Path("../data/derived")
derived_dir.mkdir(parents=True, exist_ok=True)
vol_path = derived_dir / "mru_usd_rolling_volatility.csv"
df.loc[:, ["date", "vol_30", "vol_90"]].to_csv(vol_path, index=False)
print("Saved rolling volatility to:", vol_path)

# Plot rolling volatility (interactive)
fig_vol = px.line(df, x="date", y=["vol_30", "vol_90"], title="Rolling volatility (30d, 90d)")
fig_vol.show()

# Top volatility dates
N_top = 10
top_vol30 = df[["date", "vol_30"]].dropna().nlargest(N_top, "vol_30")
top_vol90 = df[["date", "vol_90"]].dropna().nlargest(N_top, "vol_90")

print(f"Top {N_top} dates by 30-day volatility:")
display(top_vol30.reset_index(drop=True))
print(f"Top {N_top} dates by 90-day volatility:")
display(top_vol90.reset_index(drop=True))

In [ ]:
# 9. Helpers for choosing modeling period
print(f"Series date range: {date_min.date()} → {date_max.date()}")
print(f"Total observations: {n_rows}")

# observations per year
obs_per_year = df.groupby(df["date"].dt.year).size().reset_index(name="count").rename(columns={"date":"year"})
display(obs_per_year.head(12))

# observations per era
obs_per_era = df.groupby("era").size().reset_index(name="count")
display(obs_per_era)

# observations per month (sample)
df["year_month"] = df["date"].dt.to_period("M")
obs_per_month = df.groupby("year_month").size().reset_index(name="count")
display(obs_per_month.head(24))

In [ ]:
# 10. Range counting helper
def count_in_range(start_date: str, end_date: str) -> int:
    mask = (df["date"] >= pd.to_datetime(start_date)) & (df["date"] <= pd.to_datetime(end_date))
    return int(mask.sum())

candidate_ranges = [
    ("candidate_train_1", "2000-01-03", "2015-12-31"),
    ("candidate_val_1", "2016-01-01", "2019-12-31"),
    ("candidate_test_1", "2020-01-01", date_max.strftime("%Y-%m-%d")),
]

rows = []
for name, start, end in candidate_ranges:
    rows.append({"name": name, "start": start, "end": end, "count": count_in_range(start, end)})

candidate_ranges_df = pd.DataFrame(rows)
display(candidate_ranges_df)

In [ ]:
# 11. Walk-forward fold examples (for inspection)
example_folds = [
    ("F1_example", "2000-01-03", "2009-12-31", "2010-01-01", "2011-12-31"),
    ("F2_example", "2000-01-03", "2011-12-31", "2012-01-01", "2013-12-31"),
    ("F3_example", "2000-01-03", "2013-12-31", "2014-01-01", "2015-12-31"),
    ("F4_example", "2000-01-03", "2015-12-31", "2016-01-01", "2019-12-31"),
]

fold_rows = []
for name, tr_start, tr_end, va_start, va_end in example_folds:
    train_mask = (df["date"] >= pd.to_datetime(tr_start)) & (df["date"] <= pd.to_datetime(tr_end))
    val_mask = (df["date"] >= pd.to_datetime(va_start)) & (df["date"] <= pd.to_datetime(va_end))
    fold_rows.append({
        "fold": name,
        "train_start": tr_start,
        "train_end": tr_end,
        "train_count": int(train_mask.sum()),
        "val_start": va_start,
        "val_end": va_end,
        "val_count": int(val_mask.sum()),
        "overlap_train_val": bool(((df["date"] >= pd.to_datetime(va_start)) & (df["date"] <= pd.to_datetime(tr_end))).any()),
    })

fold_counts_df = pd.DataFrame(fold_rows)
display(fold_counts_df)

# Visualize fold sizes
fig_folds = px.bar(
    fold_counts_df.melt(id_vars=["fold"], value_vars=["train_count", "val_count"], var_name="set", value_name="count"),
    x="fold", y="count", color="set", barmode="group", title="Example walk-forward fold sizes"
)
fig_folds.show()

In [ ]:
# 12. Final quick sanity checks
critical_cols = ["open", "high", "low", "close", "log_return", "simple_return", "vol_30", "vol_90"]
nan_summary = df[critical_cols].isna().sum()
print("NaN counts in critical columns (note: rolling vols and returns naturally have NaNs at start):")
display(nan_summary)

